# BTC backtest readiness v1

## TL;DR

現存の実データは **confirmatory backtestには未準備**。このノートは仮説の成績を出すものではなく、保存済み公開データのcoverage、sampling bias、wallet集中、時間順splitを再現する品質監査である。許可範囲はexploratory split上のpipeline検証だけとする。

## Context and method

- raw fillsとTWAP sliceをdeduplicateしてBTC canonical fillsを作る。
- 再構成品質を通過したwalletだけを数えるが、除外率をselection biasとして明示する。
- 5/15/30/60分outcomeに必要な5分足が連続して存在するかをhorizon別に判定する。
- validationとheld-outは閾値選択に使わず、境界前60分はpurgeする。
- OI、market trades、wallet state、liquidation eventの欠測を0や推定値で補わない。

In [1]:
import csv
import json
from pathlib import Path

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "analysis/btc-backtest-v1/readiness_summary.json").exists()
)
ARTIFACT = ROOT / "analysis/btc-backtest-v1"

summary = json.loads((ARTIFACT / "readiness_summary.json").read_text(encoding="utf-8"))
splits = json.loads((ARTIFACT / "split_manifest.json").read_text(encoding="utf-8"))
with (ARTIFACT / "hypothesis_registry.csv").open(encoding="utf-8", newline="") as handle:
    hypotheses = list(csv.DictReader(handle))

summary["quality_decision"]

'NOT_READY_FOR_CONFIRMATORY_BACKTEST'

## Data readiness results

In [2]:
{
    "wallets": f'{summary["eligible_wallets"]}/{summary["sample_wallets"]}',
    "excluded_wallet_rate_pct": summary["excluded_wallet_rate_pct"],
    "completed_episodes": summary["eligible_completed_episodes"],
    "btc_completed_episodes": summary["btc_completed_episodes"],
    "btc_canonical_fills": summary["btc_canonical_fills"],
    "top5_wallet_fill_share_pct": summary["top5_wallet_btc_fill_share_pct"],
    "btc_active_5m_buckets": summary["btc_active_5m_buckets"],
}

{'wallets': '44/100',
 'excluded_wallet_rate_pct': 56.0,
 'completed_episodes': 22973,
 'btc_completed_episodes': 10125,
 'btc_canonical_fills': 69708,
 'top5_wallet_fill_share_pct': 72.7564,
 'btc_active_5m_buckets': 7291}

In [3]:
[
    {
        "horizon": horizon,
        "covered_buckets": values["buckets"],
        "coverage_pct": values["pct"],
    }
    for horizon, values in summary["outcome_coverage"].items()
]

[{'horizon': '15m', 'covered_buckets': 4070, 'coverage_pct': 55.8222},
 {'horizon': '30m', 'covered_buckets': 4067, 'coverage_pct': 55.7811},
 {'horizon': '5m', 'covered_buckets': 4072, 'coverage_pct': 55.8497},
 {'horizon': '60m', 'covered_buckets': 4061, 'coverage_pct': 55.6988}]

## Frozen chronological splits

In [4]:
{
    "splits": splits["splits"],
    "purged_at_boundaries": splits["purged_at_boundaries"],
    "status": splits["status"],
}

{'splits': [{'name': 'exploratory',
   'start': '2026-09-02T12:19:21.854000+09:00',
   'end_exclusive': '2026-09-20T23:00:00+09:00',
   'all_completed_episodes': 5144,
   'btc_completed_episodes': 952,
   'btc_active_5m_buckets': 4112,
   'independent_jst_days': 19},
  {'name': 'validation',
   'start': '2026-09-21T00:00:00+09:00',
   'end_exclusive': '2026-09-26T23:00:00+09:00',
   'all_completed_episodes': 6962,
   'btc_completed_episodes': 2059,
   'btc_active_5m_buckets': 1603,
   'independent_jst_days': 7},
  {'name': 'held_out',
   'start': '2026-09-27T00:00:00+09:00',
   'end_exclusive': '2026-10-02T11:42:13.323000+09:00',
   'all_completed_episodes': 10839,
   'btc_completed_episodes': 7105,
   'btc_active_5m_buckets': 1553,
   'independent_jst_days': 7}],
 'purged_at_boundaries': {'all_completed_episodes': 28,
  'btc_completed_episodes': 9,
  'btc_active_5m_buckets': 23},
 'status': 'BOUNDARIES_FROZEN_COUNTS_NOT_CONFIRMATORY_READY'}

## Hypothesis readiness (not performance)

In [5]:
[
    {
        "id": row["hypothesis_id"],
        "status": row["current_status"],
        "next_action": row["next_action"],
    }
    for row in hypotheses
]

[{'id': 'BTC-H01',
  'status': 'BLOCKED_CORE',
  'next_action': 'forward BTC asset contextを収集後にevent table生成'},
 {'id': 'BTC-H02',
  'status': 'DERIVABLE_NOT_BUILT',
  'next_action': 'new-entry transition aggregatorを実装しasset context待ち'},
 {'id': 'BTC-H03',
  'status': 'PARTIAL_EXPLORATORY_ONLY',
  'next_action': '連続OHLCV範囲内だけでpipeline検証。採否判断は禁止'},
 {'id': 'BTC-H04',
  'status': 'DERIVABLE_NOT_BUILT',
  'next_action': 'pilot-100 rawからoutcome-free add evidenceを生成'},
 {'id': 'BTC-H05',
  'status': 'DERIVABLE_NOT_BUILT',
  'next_action': 'past-only size baselineを実装'},
 {'id': 'BTC-H06',
  'status': 'BLOCKED_OI',
  'next_action': 'forward asset context/OI seriesが必要'},
 {'id': 'BTC-H07',
  'status': 'PARTIAL_EXPLORATORY_ONLY',
  'next_action': '連続OHLCV範囲内でoutcome pipelineだけ検証'},
 {'id': 'BTC-H08',
  'status': 'BLOCKED_NO_TRADES',
  'next_action': 'forward BTC tradesが必要'},
 {'id': 'BTC-H09',
  'status': 'PARTIAL_PANIC_ONLY',
  'next_action': 'panic-exit候補は導出可能。liquidation chainはforward event収

## Takeaways

1. 100 wallet中56 walletを除外しているため、現subsetから母集団のbehavior edgeを結論しない。
2. 上位5 walletがBTC fillの72%超を占め、fill数を独立sample数として扱わない。
3. active bucketに対する60分outcome coverageは約56%。coverage不足windowはUNAVAILABLEとする。
4. historical OI、asset context、trade stream、wallet stateがない仮説はBLOCKED。推定で埋めない。
5. 次はexploratory期間だけでH02/H04/H05のoutcome-free featureとH07 outcome pipelineを実装する。held-outは開かない。